# Day 10 — Run All Local AI Practices

Run the cells from top to bottom. Ollama must be running. This notebook uses `llama3.2:3b` for chat and `nomic-embed-text` for embeddings.

In [1]:
from pathlib import Path
import os
import sys

# Work whether Jupyter starts in the repository root or in day10.
work_dir = Path.cwd()
if not (work_dir / 'practice_01_check_ollama.py').exists():
    work_dir = work_dir / 'day10'
if not (work_dir / 'practice_01_check_ollama.py').exists():
    raise FileNotFoundError('Could not find the extracted Day 10 practice files.')
os.chdir(work_dir)

os.environ['LOCAL_LLM_MODEL'] = 'llama3.2:3b'
os.environ['LOCAL_EMBED_MODEL'] = 'nomic-embed-text'
os.environ.setdefault('OLLAMA_BASE_URL', 'http://localhost:11434')
print('Working directory:', Path.cwd())
print('Chat model:', os.environ['LOCAL_LLM_MODEL'])
print('Embedding model:', os.environ['LOCAL_EMBED_MODEL'])

Working directory: c:\Users\edgar\Documents\MainPP\AI App Dev\ai-bootcamp-portfolio\day10
Chat model: llama3.2:3b
Embedding model: nomic-embed-text


## Optional: install dependencies
Uncomment the next line if imports fail. It installs packages into this notebook's Python environment.

In [2]:
# %pip install -r requirements_day10_local_ai.txt

## Practice 01 — Check Ollama and models

In [3]:
from practice_01_check_ollama import check_ollama_server, test_chat_model, test_embedding_model

if check_ollama_server():
    test_chat_model()
    test_embedding_model()
else:
    raise RuntimeError('Start Ollama, then rerun this cell.')

✅ Ollama server is running.
Found 3 installed model(s):
 - nomic-embed-text:latest
 - qwen2.5:1.5b
 - llama3.2:3b

✅ Chat model response:
Local AI refers to the use of artificial intelligence (AI) and machine learning (ML) models on individual devices, such as smartphones or smart home appliances, to analyze and make decisions based on their own data and context.

✅ Embedding model response:
Vector length: 768
First 5 values: [0.061195966, 0.025476135, -0.18062244, -0.050456733, 0.031845927]


## Practice 02 — Load documents
Put documents in `day10/docs` before running this cell.

In [4]:
from practice_02_load_many_files import load_folder

documents = load_folder('docs')
if documents:
    print('\nPreview:', documents[0]['filename'])
    print(documents[0]['text'][:500])

✅ Loaded Day 10 Handout_ Local AI — Running Your Own LLM.html (84,822 characters)
✅ Loaded Day10__Local_AI_Slides.pdf (15,531 characters)

Total loaded files: 2

Preview: Day 10 Handout_ Local AI — Running Your Own LLM.html
<!DOCTYPE html>
<!-- saved from url=(0119)https://distance3.sg.digipen.edu/2026sg-summer/pluginfile.php/26281/mod_resource/content/4/Day10_Local_AI__Handout.html -->
<html lang="en"><head><meta http-equiv="Content-Type" content="text/html; charset=UTF-8">

<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>Day 10 Handout: Local AI — Running Your Own LLM</title>
<meta name="author" content="AI Bootcamp Faculty">
<link rel="preconnect" href="https://fonts.googleapis.com/"


## Practice 03 — Build the vector store
Set `REBUILD_VECTOR_STORE = True` when documents change. Embedding every chunk can take a while.

In [5]:
from practice_03_chunk_embed_store import build_vector_store

REBUILD_VECTOR_STORE = False
store_path = Path('local_vector_store.json')
if REBUILD_VECTOR_STORE or not store_path.exists():
    if not documents:
        raise RuntimeError('Add documents to the docs folder first.')
    build_vector_store('docs', str(store_path))
else:
    print(f'Reusing existing vector store: {store_path.resolve()}')

Reusing existing vector store: C:\Users\edgar\Documents\MainPP\AI App Dev\ai-bootcamp-portfolio\day10\local_vector_store.json


## Practice 04 — Retrieve relevant chunks

In [6]:
from practice_04_retrieve_test import retrieve

question = 'What are the main topics in my documents?'  # Change this
retrieved = retrieve(question, top_k=4)
for index, item in enumerate(retrieved, start=1):
    print(f"\n[{index}] {item['filename']} | chunk {item['chunk_index']} | score={item['score']:.3f}")
    print(item['text'][:500], '...')


[1] Day10__Local_AI_Slides.pdf | chunk 13 | score=0.534
local document Q&A After checking Ollama, each snippet adds one missing part of the RAG pipeline SUPPORTED_EXTENSIONS = {".txt", ".md", ".py", ".csv", ".html", ".pdf", ".docx"} def load_one_file(path: Path) -> dict: suffix = path.suffix.lower() if suffix in {".txt", ".md", ".py", ".csv", ".html"}: text = read_text_file(path) elif suffix == ".pdf": text = read_pdf_file(path) elif suffix == ".docx": text = read_docx_file(path) else: raise ValueError(f"Unsupported file type: {path.suffix}") return  ...

[2] Day 10 Handout_ Local AI — Running Your Own LLM.html | chunk 83 | score=0.514
">Parse .txt, .md, .py, .csv, .html, .pdf, .docx from a <code>docs/</code> folder into plain Python strings.</div> </div> <span class="toggle-icon">▼</span> </div> <div class="step-detail"> <p>Before any chunking or embedding can happen, documents need to be in plain text. Every format needs its own parser. This script normalises everything into a plain

## Practice 05 — Ask the local document assistant
Change `question` and rerun this cell as often as you like.

In [7]:
from practice_05_local_doc_qa_cli import answer_question

question = 'Summarize the most important information in my documents.'  # Change this
answer, sources = answer_question(question, top_k=4)
print('ANSWER\n')
print(answer)
print('\nSOURCES')
for index, source in enumerate(sources, start=1):
    print(f"[{index}] {source['filename']} | chunk {source['chunk_index']} | score={source['score']:.3f}")

ANSWER

Based on the provided documents, here is a summary of the most important information:

**Document Preparation**

* To use the Local AI system, documents need to be parsed into plain Python strings.
* The script uses raw libraries (e.g., `Path.read_text()`) instead of LangChain loaders to ensure transparency and control over the text output.

**Vector Database and Retrieval**

* The vector database is not the fastest but is transparent.
* The `retrieve` function retrieves the most relevant chunks by comparing the query vector with every stored chunk vector using cosine similarity.
* The `top_k` parameter determines how many top-scoring chunks to return. It means that only the best few chunks are returned, rather than all possible chunks.

**Local AI Use Cases and Benefits**

* Local AI is strongest when control and privacy matter more than absolute model strength.
* Use cases include:
	+ Sensitive data assistant
	+ Private documents
	+ Internal workflows
	+ Developer sandbox
	+ 

## Practice 06 — Launch Streamlit
Streamlit is a web server, so it runs as a child process while the notebook remains usable. Run the stop cell when finished.

In [10]:
import subprocess
import time

if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    print('Streamlit is already running at http://localhost:8501')
else:
    streamlit_process = subprocess.Popen([
        sys.executable, '-m', 'streamlit', 'run',
        'practice_06_final_streamlit_local_doc_chatbot.py',
        '--server.headless=true', '--server.port=8501'
    ])
    time.sleep(2)
    if streamlit_process.poll() is None:
        print('Streamlit started: http://localhost:8501')
    else:
        print('Streamlit exited early. Check the output above for errors.')

Streamlit started: http://localhost:8501


In [11]:
# Stop the Streamlit server when finished.
if 'streamlit_process' in globals() and streamlit_process.poll() is None:
    streamlit_process.terminate()
    streamlit_process.wait(timeout=10)
    print('Streamlit stopped.')
else:
    print('Streamlit is not running from this notebook.')

Streamlit stopped.
